# Mô hình Baseline: SARIMAX
Trong phần này, chúng ta sẽ xây dựng mô hình thống kê SARIMAX làm cơ sở (baseline).
Vì SARIMAX hoạt động tốt với dữ liệu có tính chu kỳ cố định và đơn giản, chúng ta sẽ chuyển đổi dữ liệu theo giờ (hourly) thành dữ liệu trung bình ngày (daily) để tránh mô hình bị nhiễu bởi chu kỳ phức tạp trong ngày.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# Đọc dữ liệu đúng file đã làm sạch
df = pd.read_csv('../data/processed/pm25_training_data.csv')
df['datetime'] = pd.to_datetime(df['datetime'])

# Chọn 1 thành phố đại diện (Hà Nội - nơi có ô nhiễm rõ rệt nhất)
df_hn = df[df['city'] == 'Hà Nội'].sort_values('datetime').set_index('datetime')

# Resample về trung bình ngày (Daily)
df_hourly = df_hn.copy() # Sử dụng dữ liệu theo giờ cho đồng bộ

# Bỏ các ngày bị thiếu PM2.5 (nếu có)
df_hourly = df_hourly.dropna(subset=['pm25'])

print(f"Dữ liệu sau khi gom theo ngày còn lại: {len(df_hourly)} ngày")
display(df_hourly.head(3))

Dữ liệu sau khi gom theo ngày còn lại: 32969 ngày


,pm25,pm10,o3,no2,so2,co,eu_aqi,city,temp,humidity,...,day_of_year,season,pm25_lag_1h,pm25_lag_3h,pm25_lag_6h,pm25_lag_12h,pm25_lag_24h,pm25_roll_6h,pm25_roll_24h,pm25_roll_72h
datetime,,,,,,,,,,,,,,,,,,,,,
2022-08-05 07:00:00,20.3,29.0,44.0,17.30,8.10,345.0,72.503334,Hà Nội,26.6,89.605920,...,217,Hạ,NaN,NaN,NaN,NaN,NaN,20.299999,20.299999,20.299999
2022-08-05 08:00:00,17.2,24.7,54.0,17.95,9.70,372.0,71.836670,Hà Nội,27.3,86.768300,...,217,Hạ,20.3,NaN,NaN,NaN,NaN,18.750000,18.750000,18.750000
2022-08-05 09:00:00,17.8,25.5,68.0,18.85,11.95,410.0,71.409996,Hà Nội,27.9,83.277916,...,217,Hạ,17.2,NaN,NaN,NaN,NaN,18.433333,18.433333,18.433333


## Chia tập Train/Test
Chúng ta sử dụng **Time-series split** (chia theo trình tự thời gian). 
Giữ lại 70% dữ liệu ở quá khứ để huấn luyện, 15% để kiểm định và 15% dữ liệu ở tương lai (gần nhất) để kiểm thử. Không được phép chia ngẫu nhiên (random split) đối với chuỗi thời gian.

In [2]:
# Cắt 70% thời gian đầu làm Train, 15% làm Val, 15% làm Test
train_size = int(len(df_hourly) * 0.70)
val_size = int(len(df_hourly) * 0.15)
train = df_hourly.iloc[:train_size]
val = df_hourly.iloc[train_size:train_size+val_size]
test = df_hourly.iloc[train_size+val_size:]

print(f"Tập Train: Từ {train.index.min().date()} đến {train.index.max().date()} ({len(train)} ngày)")
print(f"Tập Val:   Từ {val.index.min().date()} đến {val.index.max().date()} ({len(val)} ngày)")
print(f"Tập Test:  Từ {test.index.min().date()} đến {test.index.max().date()} ({len(test)} ngày)")

# Chọn biến mục tiêu (Target) và biến ngoại sinh (Exogenous)
target = 'pm25'
exog_cols = ['temp', 'humidity', 'wind_speed', 'pressure']

# Huấn luyện SARIMAX trên cả tập Train + Val (85%) và kiểm thử trên tập Test (15%)
train_val = pd.concat([train, val])
y_train, X_train = train_val[target], train_val[exog_cols]
y_test, X_test = test[target], test[exog_cols]

Tập Train: Từ 2022-08-05 đến 2025-03-23 (23078 ngày)
Tập Val:   Từ 2025-03-23 đến 2025-10-15 (4945 ngày)
Tập Test:  Từ 2025-10-15 đến 2026-05-09 (4946 ngày)


## Huấn luyện và Đánh giá SARIMAX
Sử dụng mô hình SARIMAX với các tham số dự đoán mùa vụ và kết hợp thêm các biến ngoại sinh.

In [ ]:
print("Đang huấn luyện SARIMAX (quá trình này có thể mất vài phút)...")

# Cấu hình SARIMAX(p,d,q)(P,D,Q,s)
# s=24 để mô hình hiểu chu kỳ 24 giờ trong ngày
model = SARIMAX(
    y_train,
    exog=X_train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 24)
)
results = model.fit(disp=False)

# Dự đoán trên tập Test
predictions = results.predict(
    start=len(train),
    end=len(train) + len(test) - 1,
    exog=X_test
)

# Tính toán sai số
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)

print(f"\n📊 KẾT QUẢ SARIMAX BASELINE:")
print(f"   RMSE: {rmse:.2f} µg/m³")
print(f"   MAE:  {mae:.2f} µg/m³")

# Trực quan hóa kết quả
plt.figure(figsize=(14, 5))
plt.plot(y_test.index, y_test.values, label='Thực tế (PM2.5)', alpha=0.7)
plt.plot(y_test.index, predictions.values, color='red', label='Dự đoán SARIMAX', alpha=0.8)
plt.title('SARIMAX Baseline: Dự báo PM2.5 tại Hà Nội')
plt.ylabel('PM2.5 (µg/m³)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Đang huấn luyện SARIMAX (quá trình này có thể mất vài phút)...
